# Batch Normalisation

> Based on Stanford CS230 — C2M3, Andrew Ng / DeepLearning.AI

Batch normalisation (BN) normalises the pre-activations $Z^{[l]}$ within each mini-batch, then re-scales them with learned parameters $\gamma$ and $\beta$. This stabilises training, allows higher learning rates, and acts as a mild regulariser.

## Learning Objectives

1. Write the full BN forward and backward pass
2. Understand the difference between training and inference behaviour
3. Visualise covariate shift and how BN addresses it
4. Know where BN fits in the layer stack relative to activation functions

## Batch Norm Forward Pass

For a mini-batch of pre-activations $Z^{[l]} \in \mathbb{R}^{n \times m}$:

$$\mu_B = \frac{1}{m}\sum_{i=1}^m z^{(i)}, \qquad \sigma_B^2 = \frac{1}{m}\sum_{i=1}^m (z^{(i)} - \mu_B)^2$$

$$\hat{z}^{(i)} = \frac{z^{(i)} - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \qquad \tilde{z}^{(i)} = \gamma \hat{z}^{(i)} + \beta$$

Learnable parameters $\gamma, \beta \in \mathbb{R}^{n}$ allow the network to undo normalisation if needed.

**At inference time** use running averages $\mu_{\text{run}}, \sigma^2_{\text{run}}$ accumulated during training (not the batch statistics).


## Position in the Layer Stack

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 680 110" width="680" height="110" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <defs>
    <marker id="a2" markerWidth="7" markerHeight="7" refX="5" refY="3" orient="auto">
      <path d="M0,0 L0,6 L7,3 z" fill="#888"/>
    </marker>
  </defs>
  <!-- Boxes -->
  <rect x="20"  y="30" width="100" height="50" rx="5" fill="#5b9bd5" fill-opacity="0.15" stroke="#5b9bd5" stroke-width="1.6"/>
  <rect x="170" y="30" width="100" height="50" rx="5" fill="#f4b942" fill-opacity="0.15" stroke="#f4b942" stroke-width="1.6"/>
  <rect x="320" y="30" width="120" height="50" rx="5" fill="#c678dd" fill-opacity="0.15" stroke="#c678dd" stroke-width="1.6"/>
  <rect x="490" y="30" width="100" height="50" rx="5" fill="#7ecba1" fill-opacity="0.15" stroke="#7ecba1" stroke-width="1.6"/>
  <!-- Labels -->
  <text x="70"  y="53" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">Linear</text>
  <text x="70"  y="68" text-anchor="middle" fill="#3a7bbf" font-size="10">Z = WA + b</text>
  <text x="220" y="53" text-anchor="middle" fill="#a07010" font-size="11" font-weight="bold">Batch Norm</text>
  <text x="220" y="68" text-anchor="middle" fill="#a07010" font-size="10">Z̃ = γẑ + β</text>
  <text x="380" y="53" text-anchor="middle" fill="#8a3ab8" font-size="11" font-weight="bold">Activation</text>
  <text x="380" y="68" text-anchor="middle" fill="#8a3ab8" font-size="10">A = g(Z̃)</text>
  <text x="540" y="53" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Next Layer</text>
  <text x="540" y="68" text-anchor="middle" fill="#3d7a5e" font-size="10">...</text>
  <!-- Arrows -->
  <line x1="120" y1="55" x2="167" y2="55" stroke="#888" stroke-width="1.4" marker-end="url(#a2)"/>
  <line x1="270" y1="55" x2="317" y2="55" stroke="#888" stroke-width="1.4" marker-end="url(#a2)"/>
  <line x1="440" y1="55" x2="487" y2="55" stroke="#888" stroke-width="1.4" marker-end="url(#a2)"/>
  <!-- Annotations -->
  <text x="70"  y="96" text-anchor="middle" fill="#888" font-size="9">b redundant with β</text>
  <text x="220" y="96" text-anchor="middle" fill="#888" font-size="9">normalise + rescale</text>
  <text x="380" y="96" text-anchor="middle" fill="#888" font-size="9">ReLU / sigmoid</text>
</svg>

> Note: because BN includes its own shift parameter $\beta$, the bias term $b$ in the linear layer becomes redundant and can be omitted.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Batch Norm implementation ----
class BatchNorm:
    def __init__(self, n, momentum=0.9, eps=1e-8):
        self.gamma = np.ones((n, 1))
        self.beta  = np.zeros((n, 1))
        self.eps   = eps
        self.momentum = momentum
        self.running_mean = np.zeros((n, 1))
        self.running_var  = np.ones((n, 1))
        self.cache = None

    def forward(self, Z, training=True):
        if training:
            mu    = Z.mean(axis=1, keepdims=True)
            var   = Z.var(axis=1,  keepdims=True)
            Z_hat = (Z - mu) / np.sqrt(var + self.eps)
            Z_tilde = self.gamma * Z_hat + self.beta
            self.cache = (Z, Z_hat, mu, var)
            self.running_mean = self.momentum * self.running_mean + (1-self.momentum) * mu
            self.running_var  = self.momentum * self.running_var  + (1-self.momentum) * var
        else:
            Z_hat   = (Z - self.running_mean) / np.sqrt(self.running_var + self.eps)
            Z_tilde = self.gamma * Z_hat + self.beta
        return Z_tilde

    def backward(self, dZ_tilde, lr=0.01):
        Z, Z_hat, mu, var = self.cache
        m = Z.shape[1]
        d_gamma = (dZ_tilde * Z_hat).sum(axis=1, keepdims=True)
        d_beta  = dZ_tilde.sum(axis=1, keepdims=True)
        dZ_hat  = dZ_tilde * self.gamma
        std_inv = 1 / np.sqrt(var + self.eps)
        dZ = (1/m) * std_inv * (m * dZ_hat
                                 - dZ_hat.sum(axis=1, keepdims=True)
                                 - Z_hat * (dZ_hat * Z_hat).sum(axis=1, keepdims=True))
        self.gamma -= lr * d_gamma
        self.beta  -= lr * d_beta
        return dZ

# ---- Visualise: activation shift (covariate shift) ----
np.random.seed(0)
n_units, m_batch = 64, 256

# Simulate activations at two different time points
Z1 = np.random.randn(n_units, m_batch) * 1.0 + 0.0   # initial distribution
Z2 = np.random.randn(n_units, m_batch) * 3.0 + 5.0   # shifted distribution (covariate shift)

bn = BatchNorm(n_units)
Z1_bn = bn.forward(Z1, training=True)
Z2_bn = bn.forward(Z2, training=True)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Batch Normalisation — Effect on Activation Distributions', fontsize=13, fontweight='bold')

# Before BN
axes[0][0].hist(Z1.flatten(), bins=50, color=P[0], alpha=0.75, label='Batch 1  (μ=0, σ=1)')
axes[0][0].hist(Z2.flatten(), bins=50, color=P[1], alpha=0.75, label='Batch 2  (μ=5, σ=3) — shifted')
axes[0][0].set_title('Before Batch Norm')
axes[0][0].set_xlabel('Activation value')
axes[0][0].legend(fontsize=9)
axes[0][0].grid(True)

# After BN
axes[0][1].hist(Z1_bn.flatten(), bins=50, color=P[0], alpha=0.75, label='Batch 1 after BN')
axes[0][1].hist(Z2_bn.flatten(), bins=50, color=P[1], alpha=0.75, label='Batch 2 after BN')
axes[0][1].set_title('After Batch Norm  (both normalised to ≈ N(0,1))')
axes[0][1].set_xlabel('Activation value')
axes[0][1].legend(fontsize=9)
axes[0][1].grid(True)

# ---- Training comparison with / without BN ----
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z): return np.maximum(0, z)

def train_with_bn(X, Y, use_bn=True, lr=0.1, n_epochs=200, seed=1):
    np.random.seed(seed)
    n0, m = X.shape
    n_h = 32
    W1 = np.random.randn(n_h, n0) * np.sqrt(2/n0)
    W2 = np.random.randn(1, n_h)  * np.sqrt(2/n_h)
    b2 = np.zeros((1, 1))
    bn_layer = BatchNorm(n_h) if use_bn else None
    costs = []
    for ep in range(n_epochs):
        Z1 = W1 @ X
        A1 = relu(bn_layer.forward(Z1) if use_bn else Z1)
        Z2 = W2 @ A1 + b2
        AL = sigmoid(Z2)
        cost = -np.mean(Y*np.log(AL+1e-8) + (1-Y)*np.log(1-AL+1e-8))
        costs.append(cost)
        dAL = -(Y/(AL+1e-8) - (1-Y)/(1-AL+1e-8))
        dZ2 = dAL * AL*(1-AL)
        dW2 = (dZ2 @ A1.T)/m
        db2 = np.mean(dZ2, axis=1, keepdims=True)
        dA1 = W2.T @ dZ2
        dZ1_pre = dA1 * (Z1 > 0).astype(float)
        if use_bn:
            dZ1_pre = bn_layer.backward(dZ1_pre, lr)
        dW1 = (dZ1_pre @ X.T)/m
        W1 -= lr * dW1; W2 -= lr * dW2; b2 -= lr * db2
    return costs

np.random.seed(3)
m_tr = 500
X_tr = np.random.randn(8, m_tr)
Y_tr = (np.random.rand(1, m_tr) > 0.5).astype(float)

costs_bn  = train_with_bn(X_tr, Y_tr, use_bn=True,  lr=0.3, n_epochs=400)
costs_no  = train_with_bn(X_tr, Y_tr, use_bn=False, lr=0.3, n_epochs=400)
costs_no_slow = train_with_bn(X_tr, Y_tr, use_bn=False, lr=0.05, n_epochs=400)

axes[1][0].plot(costs_bn,      color=P[3], lw=2.2, label='With BN  (lr=0.3)')
axes[1][0].plot(costs_no,      color=P[1], lw=2.2, label='Without BN  (lr=0.3)')
axes[1][0].plot(costs_no_slow, color=P[0], lw=2.2, ls='--', label='Without BN  (lr=0.05)')
axes[1][0].set_xlabel('Epoch'); axes[1][0].set_ylabel('Cost')
axes[1][0].set_title('Training with vs without Batch Norm')
axes[1][0].legend(fontsize=9); axes[1][0].grid(True)

# γ and β evolution (track over training iterations)
np.random.seed(5)
n_units2, m2 = 16, 128
bn2 = BatchNorm(n_units2)
gammas, betas = [], []
Z_input = np.random.randn(n_units2, m2) * 3 + 2

for step in range(100):
    Z_out = bn2.forward(Z_input + np.random.randn(n_units2, m2)*0.5)
    dZ_out = np.random.randn(*Z_out.shape) * 0.1
    bn2.backward(dZ_out, lr=0.05)
    gammas.append(bn2.gamma.mean())
    betas.append(bn2.beta.mean())

axes[1][1].plot(gammas, color=P[0], lw=2, label='γ (scale) — mean across units')
axes[1][1].plot(betas,  color=P[1], lw=2, label='β (shift) — mean across units')
axes[1][1].axhline(1, color='#ccc', lw=1, ls='--')
axes[1][1].axhline(0, color='#ccc', lw=1, ls='--')
axes[1][1].set_xlabel('Training step')
axes[1][1].set_title('γ and β Parameters During Training')
axes[1][1].legend(fontsize=9); axes[1][1].grid(True)

plt.tight_layout()
plt.show()
